# Tutorial 3: Learn about `Selenium`

Follow along with this [RealPython tutorial on Selenium](https://realpython.com/modern-web-automation-with-python-and-selenium/).

**Crystal Zhao**

Purpose of Selenium: whe modern sites rely on JavaScript to generate content dynamically, an HTTP request will not reveal the full page content. We need Selenium to:
- lauch a visible browser using a web driver
- visit URLs and navigate pages like a real user
- locate elements with CSS selectors / XPath / other locators
- interact with elements by clicking, typing, dragging, or waiting for them to chage

Selenium is written in Java 

Page Object Model (not document object model):
Represent each web page or component in an application by the dedicated class of page object.

### 1. Navigate a web page with Python and Selenium

Always interact with a page first before committing to anything.

**Launch a headless browser and navigate to URL**

I had to install Firefox.

In [4]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options

options = Options()
options.add_argument("--headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)

driver.get("https://bandcamp.com/discover/")
print(driver.title)

driver.quit()

Discover music and merch on Bandcamp


**Locating elements in the DOM**
- by ID
- by CSS selector
- by XPath
- by link text, tag name, or class name 

1. single element: find_element (by ID)
2. all track elements: find_elements(by class name)

In [5]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By

options = Options()
options.add_argument("--headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)

driver.get("https://bandcamp.com/discover/")
print(driver.title)

pagination_button = driver.find_element(By.ID, "view-more")
print(pagination_button.accessible_name)

tracks = driver.find_elements(By.CLASS_NAME, "results-grid-item")
print(len(tracks))
print(tracks[0].text)

driver.quit()

Discover music and merch on Bandcamp
View more results
60
electronic
News At 11 [10th Anniversary Edition]
by 猫 シ Corp.


Two options: find_element (ID) or find elements (class name); 
NEED to have already inspected thoroughly so we could identify locators to target elements. 

### 2. Interact with Web Elements

**Click buttons and links**

In [8]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By

options = Options()
options.add_argument("--headless")
driver = webdriver.Firefox(options=options)
driver.implicitly_wait(5)

driver.get("https://bandcamp.com/discover/")

tracks = driver.find_elements(By.CLASS_NAME, "results-grid-item")
print(len(tracks))

driver.quit()

60


We have identified the full page and now we would want to see more results (scroll on).

In [9]:
import time

# ...

pagination_button = driver.find_element(By.ID, "view-more")
pagination_button.click()

time.sleep(0.5)

tracks = driver.find_elements(By.CLASS_NAME, "results-grid-item")
print(len(tracks))

driver.quit()

MaxRetryError: HTTPConnectionPool(host='localhost', port=61600): Max retries exceeded with url: /session/186ae144-76f1-4851-a3b1-3d520a69601b/element (Caused by NewConnectionError("HTTPConnection(host='localhost', port=61600): Failed to establish a new connection: [Errno 61] Connection refused"))

**Send Keystrokes and Text Entry**

dealing with input fields using send_keys (typing text into them) 

In [12]:
import time
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By

driver = webdriver.Firefox()  # Run in normal mode
driver.implicitly_wait(5)

driver.get("https://bandcamp.com/discover/")

# Accept cookies, if required
try:
    cookie_accept_button = driver.find_element(
        By.CSS_SELECTOR,
        "#cookie-control-dialog button.g-button.outline",
    )
    cookie_accept_button.click()
except NoSuchElementException: # if this doesn't exist, we just continue normal execution
    pass

time.sleep(0.5)

search = driver.find_element(By.CLASS_NAME, "site-search-form")
search_field = search.find_element(By.TAG_NAME, "input")
search_field.send_keys("selenium")
search_field.submit()

time.sleep(5)

driver.quit()

**Hideen or Overlaid Elements**
For example, a cookie overlay (normal part of the page) 

**Use Hover, drag and drop, and more complex gestures**

ActionChains class

Identify both relevant elements and use "move_to_element()": this performs a hover action on the selected elements, triggering a drop down which allows you to select a sub-menu item --> now you can interact by clicking 

**Submit Forms**

Continue using sed_keys: fill in multiple input fields

Might trigger a page load or AJAX call --> pair with wait condition 

### 3. Handle Dynamic Content

Must wait until the content is actually here; 

time.sleep() is only one option

**Understand Implicit, Explicit, and Fluent Waits**

1. Implicit Wait: Selenium polls the DOM for a specified time whenever you try to find an element


In [13]:
driver.implicitly_wait(5)

MaxRetryError: HTTPConnectionPool(host='localhost', port=63089): Max retries exceeded with url: /session/8ede9223-62c2-4cfc-800d-c9e1f31ef788/timeouts (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63089): Failed to establish a new connection: [Errno 61] Connection refused"))

2. Explicit wait: Seleium applies conditions to waits, which makes it more flexible

Use WebDriverWait object in combination with predefined conditions
Example: View More Results button is not available as the page is actively loading

In [15]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)
wait.until( # literally, wait until this is clickable  
    EC.element_to_be_clickable((By.ID, 'view-more') # this returns the element that Selenium was waiting for 
))

MaxRetryError: HTTPConnectionPool(host='localhost', port=63089): Max retries exceeded with url: /session/8ede9223-62c2-4cfc-800d-c9e1f31ef788/element (Caused by NewConnectionError("HTTPConnection(host='localhost', port=63089): Failed to establish a new connection: [Errno 61] Connection refused"))

There are many different expected conditions, like waiting for an element to be present, to be visible, to be clickable, to hae an alert...

**Sychronization issues**

Unstable issues;
E.g., if running on headless browsers, some things run quicker
E.g., JavaScript alerts

3. Fluent wait: you specify the polling interval ad we ignore certain exceptions that occur during polling 

### 4. Implement the Page Object Model (POM)

PROBLEM: cannot pile all code into one file because there is a mess of locators, wait statemenets, adn repeated logic...

SOLUTION: POM design pattern - represent each significant page component with a dedicated class in your codde

1. Create a new class for each separate web page --> pages.py

2. Focus primarily on page elements, or panels, than on full pages

**Establish a base for pages and elements**
Result: base.py

Setuplogic: webpage (contstants like viewport size, initialize a webdriverwait object); webcomponent (reference to a parent webelement)

**Describe webpage as a page object**

Can omit things that you do not need! Need to carefully define objectives:
Play music -> tracks, cookie consent form 

class DiscoverPage(WebPage): # reuse what is set up in base.py, inherit from it
1. def _init_
2. def _accept_cookie_consent(self)

**Define reusable web elements** --> results: elements.py

1. TrackListElement: container element
2. TrackElement: single track

class TrackListElement(WebComponent):
- def load_more(self)
- def _get_available_tracks(self)


class TrackElemet(WebComponent):
- def play(self)
- def pause(self)
- def _get_track_info(self)

The track list is now a page object, and the track element is also modeled (including the important interactions).

**Keep Locators Separate**

Locators can quickly change, so keep locators in a dedicated file.

class DiscoverPageLocator:
    Discover_results = (By.CLASS_NAME, 'results-grid') # this is a locator

Not necessary to wrap locators into classes, they could also be global constants.